In [ ]:
import pandas as pd
import numpy as np

import lightgbm as lgb
from sklearn.ensemble import HistGradientBoostingClassifier
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score, precision_recall_curve, auc
from sklearn.utils.class_weight import compute_sample_weight

## Import des données

In [ ]:
df = pd.read_csv("../../data/resample_normalized_flagged_v3.csv")

## Préparation des données

In [ ]:
df = df.drop(columns=["apogee_id", "STARFLAGS"])

class_columns = ['class_spectral', 'class_lum_logg', 'class_lum_jhk', 'class_lum_bins_logg', 'class_lum_bins_jhk']

chemical_columns = ['C_FE', 'CI_FE', 'N_FE', 'O_FE', 'NA_FE', 'MG_FE', 'AL_FE', 'SI_FE', 'S_FE', 'K_FE', 'CA_FE', 'TI_FE', 'V_FE', 'CR_FE', 'MN_FE', 'NI_FE', 'FE_H']
physique_columns = ['J', 'H', 'K', 'LOGG', 'M_H', 'VMICRO', 'VMACRO']

df_chem = df.drop(columns=physique_columns)
df_phys = df.drop(columns=chemical_columns)

In [ ]:
target_column = 'class_spectral'
X = df_chem.drop(columns=class_columns)
y = df_chem[target_column]

In [ ]:
# Calcul des fréquences des classes
frequencies = df_chem[target_column].value_counts(normalize=True)

# Calcul des poids inverses
weights = (1 / frequencies).to_dict()

### Implémentation de LightGBM

In [ ]:
# Division en ensemble d'entraînement et de test
X_train_LGB, X_test_LGB, y_train_LGB, y_test_LGB = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

lgb_model = lgb.LGBMClassifier(n_estimators=2000, learning_rate=0.05, class_weight=weights, random_state=42, objective="multiclass", num_class=3)

lgb_model.fit(X_train_LGB, y_train_LGB, eval_set=[(X_test_LGB, y_test_LGB)])

y_pred_LGB = lgb_model.predict(X_test_LGB)

In [ ]:
#évaluation du modèle
print("Accuracy : \n", accuracy_score(y_test_LGB, y_pred_LGB))
print("Classification Report : \n", classification_report(y_test_LGB, y_pred_LGB))

In [ ]:
f1_LGB = f1_score(y_test_LGB, y_pred_LGB, average="weighted")  # "weighted" pour prendre en compte le déséquilibre
print("F1 Score:", f1_LGB)

In [ ]:
y_probs_LGB = lgb_model.predict_proba(X_test_LGB)  # Probabilités prédites des classes

auc_pr_list_LGB = []
for i in range(len(np.unique(y_test_LGB))):
    precision, recall, _ = precision_recall_curve((y_test_LGB == i).astype(int), y_probs_LGB[:, i])
    auc_pr_list_LGB.append(auc(recall, precision))
auc_pr_LGB = np.mean(auc_pr_list_LGB)  # Moyenne sur toutes les classes
print("AUC-PR (moyenne sur classes):", auc_pr_LGB)

### Implémentation de HistGradientBoosting

In [ ]:
# Division en ensemble d'entraînement et de test
X_train_HGB, X_test_HGB, y_train_HGB, y_test_HGB = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

sample_weights = compute_sample_weight(class_weight=weights, y=y_train_HGB)

# Initialisation du modèle
hgb = HistGradientBoostingClassifier(loss="log_loss", learning_rate=0.1, max_iter=100)

# Entraînement
hgb.fit(X_train_HGB, y_train_HGB, sample_weight=sample_weights)

# Prédiction
y_pred_HGB = hgb.predict(X_test_HGB)

In [ ]:
# Évaluation
print("Accuracy:", accuracy_score(y_test_HGB, y_pred_HGB))
print("Classification Report:\n", classification_report(y_test_HGB, y_pred_HGB))

In [ ]:
f1_HGB = f1_score(y_test_HGB, y_pred_HGB, average="weighted")  # "weighted" pour prendre en compte le déséquilibre
print("F1 Score:", f1_HGB)

In [ ]:
y_probs_HGB = hgb.predict_proba(X_test_HGB)  # Probabilités prédites des classes

auc_pr_list_HGB = []
for i in range(len(np.unique(y_test_HGB))):
    precision, recall, _ = precision_recall_curve((y_test_HGB == i).astype(int), y_probs_HGB[:, i])
    auc_pr_list_HGB.append(auc(recall, precision))
auc_pr_HGB = np.mean(auc_pr_list_HGB)  # Moyenne sur toutes les classes
print("AUC-PR (moyenne sur classes):", auc_pr_HGB)

### Implémentation de CatBoost

In [ ]:
# Division en ensemble d'entraînement et de test
X_train_CB, X_test_CB, y_train_CB, y_test_CB = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Création et entraînement du modèle
model_catboost = CatBoostClassifier(iterations=1000, learning_rate=0.05, depth=6, loss_function='MultiClass',  eval_metric='MultiClass', class_weights=weights, verbose=100)

model_catboost.fit(X_train_CB, y_train_CB, eval_set=(X_test_CB, y_test_CB), early_stopping_rounds=100)

# Prédictions
y_pred = model_catboost.predict(X_test_CB)

In [ ]:
# Performance
print("Accuracy:", accuracy_score(y_test_CB, y_pred))
print(classification_report(y_test_CB, y_pred))

In [ ]:
f1_CB = f1_score(y_test_CB, y_pred, average="weighted")  # "weighted" pour prendre en compte le déséquilibre
print("F1 Score:", f1_CB)

In [ ]:
y_probs_CB = model_catboost.predict_proba(X_test_CB)  # Probabilités prédites des classes

auc_pr_list_CB = []
for i in range(len(np.unique(y_test_CB))):
    precision, recall, _ = precision_recall_curve((y_test_CB == i).astype(int), y_probs_CB[:, i])
    auc_pr_list_CB.append(auc(recall, precision))
auc_pr_CB = np.mean(auc_pr_list_CB)  # Moyenne sur toutes les classes
print("AUC-PR (moyenne sur classes):", auc_pr_CB)